# CSCI1409 Deep Learning, Week 7 SIS Mini-project 2
## CIFAR-10: your own CNN, Dropout, and why accuracy is not enough

SIS mini-project 2: 4 course points · Defense: in class, Week 7

This lab covers both Week 7 topics: regularization (Dropout) and model evaluation metrics.

Goal: design and code your own CNN for color images, train it twice (without and with Dropout), and check how good it really is with more than accuracy: precision, recall, F1, average precision (AP) and a confusion matrix. At the end you tune the decision threshold for the two rare classes.

One twist: in this lab cats and dogs are rare. Imagine an app where most photos show planes, cars, ships, birds or horses, and only a few show cats or dogs. You will find out how well your models work on these rare classes, and how much a better threshold can help.

The most important rule of this lab: every decision (which model is best, which threshold to use) is made on the validation set. The test set is used once, in section 8, after all decisions are made.

You work in groups of 3. Apart from the dataset download, all code in this notebook is written by your group. No ready-made or pretrained networks: you build the CNN yourselves from PyTorch layers. The cells only contain `# TODO` comments with hints. Every student must be able to explain every line and every plot. Write short answers in the answer cells (2 to 4 sentences each is enough), and every answer must use your own numbers.

Before class: make sure `torch`, `datasets` (Hugging Face), `matplotlib` and `scikit-learn` are installed ([PyTorch setup guide](https://pytorch.org/get-started/locally/), then `pip install datasets matplotlib scikit-learn`). The first download of CIFAR-10 from Hugging Face (about 144 MB) needs internet. A CPU is enough, no GPU needed. Training both models for 15 epochs takes roughly 10 to 30 minutes on a laptop CPU, depending on the size of your CNN, so do not start the night before. A GPU (for example free Google Colab) makes it faster but is not required.

### AI-use rule for this lab (Level C, AI-Assisted)

You may use generative AI to get ideas, explain concepts, debug code, or get feedback. However, your group must design the CNN and write the main solution yourselves, and understand and check everything you submit. At the defense you will explain your code, plots and metrics without AI help. "The AI wrote it" is not an answer.

Check AI explanations against the lectures and the PyTorch and scikit-learn documentation. A general definition without your own numbers does not count as an answer in this lab. Plagiarism, copying another group's code, and invented results are not allowed under any circumstances. If you used an external source (tutorial, blog, docs), name it in your answers.

## 1. Dataset

CIFAR-10 has 32×32 color images in 10 classes: airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck. The download code is given: it loads the dataset [uoft-cs/cifar10](https://huggingface.co/datasets/uoft-cs/cifar10) from Hugging Face and gives you `full_train` (50,000 images) and `full_test` (10,000 images). Both work like `torchvision.datasets.CIFAR10`:
- `full_train[i]` gives `(image, label)`, where `image` is a float tensor of shape `[3, 32, 32]` with values from 0 to 1, and `label` is an int from 0 to 9
- `full_train.targets` is a list with the label of every image
- `full_train.classes` is the list of class names, in label order

You build three sets with the sizes below. Cats (class 3) and dogs (class 5) get 10 times fewer images than the other classes, in all three sets.

| Set | Taken from | Each of the 8 other classes | Cat | Dog | Total |
|---|---|---|---|---|---|
| Train | official training portion | 1,000 | 100 | 100 | 8,200 |
| Validation | official training portion, images not used in train | 500 | 50 | 50 | 4,100 |
| Test | official test set | 1,000 | 100 | 100 | 8,200 |

The validation set is bigger than you may expect. In section 7 you use it to choose thresholds for cats and dogs, and with only a few cats that choice would be very noisy.

Key words:
- Training set: the images the model learns from.
- Validation set: images used to check the model during training and to make decisions (best model, thresholds).
- Test set: images used only once, at the very end, after all decisions are made. In this lab that is section 8.

In [1]:
# Given code: download CIFAR-10 from Hugging Face (uoft-cs/cifar10). You do not need to change this cell.
import numpy as np
import torch
from torch.utils.data import Dataset
from datasets import load_dataset

DATA_ROOT = "./data"


class CIFAR10FromHF(Dataset):
    # One CIFAR-10 split. Works like torchvision.datasets.CIFAR10 with transforms.ToTensor().

    def __init__(self, hf_split):
        # decode all images once into one uint8 array of shape [N, 32, 32, 3] (takes up to a minute)
        self.images = np.stack([np.array(img.convert("RGB")) for img in hf_split["img"]])
        self.targets = list(hf_split["label"])            # one int label per image
        self.classes = hf_split.features["label"].names   # class names in label order

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, i):
        # [32, 32, 3] uint8 in 0..255  ->  [3, 32, 32] float in 0..1
        image = torch.from_numpy(self.images[i]).permute(2, 0, 1).float() / 255.0
        return image, self.targets[i]


hf = load_dataset("uoft-cs/cifar10", cache_dir=DATA_ROOT)  # the first run downloads about 144 MB
full_train = CIFAR10FromHF(hf["train"])
full_test = CIFAR10FromHF(hf["test"])

print("Official training portion:", len(full_train), "images")
print("Official test set:", len(full_test), "images")
print("Classes:", full_train.classes)

ModuleNotFoundError: No module named 'datasets'

In [ ]:
# TODO: import what you need (torch, torch.nn, DataLoader, Subset, matplotlib, numpy, sklearn.metrics)
# TODO: fix a random seed (e.g. torch.manual_seed(42)) so your results are repeatable
# TODO: pick the device: "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
# TODO: build the train and val indices from full_train using the table above
#       hint: full_train.targets is a list with the label of every image
#       hint: for each class, shuffle its indices with torch.randperm, then take the first N for train and the next M for val
# TODO: build the test indices from full_test the same way
# TODO: wrap the indices with torch.utils.data.Subset and print the size of each set

In [ ]:
# TODO: count and print how many images of each class are in train, val and test, and check they match the table
# TODO: show one training image per class with its class name
#       hint: matplotlib needs the shape [32, 32, 3], so use image.permute(1, 2, 0)

#### Your answers: dataset

- What does the shape `[3, 32, 32]` mean? How is it different from FashionMNIST?
- What share of the test set are cats and dogs together?
- Why do we use the test set only once, at the end?

Write your answers here.

## 2. Design your CNN

You design one CNN architecture and train it twice:
- Model A: your CNN without Dropout.
- Model B: exactly the same CNN with Dropout added. Nothing else changes.

Rules for your CNN:

| Rule | Requirement |
|---|---|
| Written by you | Build it from PyTorch layers: `nn.Conv2d`, `nn.ReLU`, `nn.MaxPool2d`, `nn.Flatten`, `nn.Linear`, `nn.Dropout`. Not allowed: `torchvision.models`, `timm`, pretrained weights, or a network copied from a tutorial. |
| Depth | 5 to 10 layers with weights. Count `Conv2d` and `Linear` layers together. ReLU, MaxPool, Flatten and Dropout have no weights, so they do not count. |
| Convolution | At least 3 `Conv2d` layers, each followed by `ReLU`. |
| Pooling | At least 2 `MaxPool2d` layers. |
| Classifier head | `Flatten`, then at least one hidden `Linear` layer with `ReLU`, then `Linear(…, 10)`. |
| Output | 10 logits. No softmax inside the model: `nn.CrossEntropyLoss` applies it for you. |

`nn.BatchNorm2d` is allowed if you can explain what it does, but it is not required.

An example that follows the rules (6 layers with weights). You may use it or design your own:

- Block 1: `Conv2d(3, 32, kernel_size=3, padding=1) → ReLU → Conv2d(32, 32, kernel_size=3, padding=1) → ReLU → MaxPool2d(2)`
- Block 2: `Conv2d(32, 64, kernel_size=3, padding=1) → ReLU → Conv2d(64, 64, kernel_size=3, padding=1) → ReLU → MaxPool2d(2)`
- Head: `Flatten → Linear(?, 256) → ReLU → Linear(256, 10)`

Finding the number in place of `?` for your own CNN is part of the task.

Dropout for Model B:
- Required: `nn.Dropout(p=0.5)` right after the hidden `ReLU`, before the output `Linear`.
- Recommended: also `nn.Dropout(p=0.25)` right after `Flatten`.
- You may try other places or values of `p`, but choose using the validation set and write down what you tried.

In [ ]:
# TODO: write a function make_cnn(use_dropout) that builds your CNN as an nn.Module (or nn.Sequential)
#       use_dropout=False -> Model A, use_dropout=True -> Model B
#       one function for both models makes sure they cannot differ by accident
#       hint: nn.Conv2d, nn.ReLU, nn.MaxPool2d, nn.Flatten, nn.Linear, nn.Dropout

In [ ]:
# TODO: pass a small batch through Model A and Model B and check the output shape is (batch_size, 10)
# TODO: print the output shape after every layer of your CNN
#       hint: loop over the layers of an nn.Sequential and apply them one by one
# TODO: count the layers with weights (Conv2d + Linear) and check every rule from the table
# TODO: count the trainable parameters of both models, and separately the parameters in conv layers and in linear layers
#       hint: sum(p.numel() for p in model.parameters())

#### Your answers: your CNN

- Write your architecture as a list of layers with the output shape after each one. How many layers with weights does it have?
- What number did you put in place of `?`, and how did you get it? Show how height and width change after every `Conv2d` and `MaxPool2d`.
- How many parameters does your CNN have in total, in the conv layers, and in the linear layers? Compute the parameters of one `Conv2d` layer by hand and check it against your code.
- Why do the conv layers need so few parameters? Think about what a 3×3 filter looks at, and that the same filter slides over the whole image.
- Why do Model A and Model B have exactly the same number of parameters? Why is there no softmax at the end of your model?

Write your answers here.

## 3. Training setup

Use these settings for both models (you may change them, but write down what you used):

| Setting | Value |
|---|---|
| Optimizer | Adam |
| Learning rate | 0.001 |
| Batch size | 64 |
| Epochs | 15 |

Keep the seed, optimizer, learning rate, batch size and number of epochs the same for both models. Only Dropout changes, so the comparison is fair. We use 15 epochs so that overfitting has time to show up.

In [ ]:
# TODO: create DataLoaders for train (shuffle=True), val and test (shuffle=False)
# TODO: choose a loss function for 10-class classification

#### Your prediction (write it before training)

- Which model will reach the higher validation accuracy, and why?
- Which model will overfit more? What do you expect Dropout to change in the loss curves?
- Which classes do you think will be predicted worst?

Write your prediction here.

## 4. Training and evaluation code

You can reuse `train_one_epoch` and `evaluate` from SIS 1. Changes: move every batch to your device (`images.to(device)`, `labels.to(device)`), move each model to the device after you build it, and switch between train and eval mode correctly, because Dropout depends on it.

Key words:
- Overfitting: the training loss keeps going down, but the validation loss stops improving or goes up. The model memorises the training images instead of learning patterns that also work on new images.
- Regularization: any method that reduces overfitting. Dropout is one of them.
- Dropout with probability `p`: during training, every value that enters the Dropout layer is set to zero with probability `p`, independently for every image and every step. The network cannot rely on one single neuron, so it learns features that still work when some neurons are missing (Goodfellow et al., *Deep Learning*, Section 7.12).
- Inverted dropout: during training, PyTorch multiplies the values that are kept by `1 / (1 − p)`. Because of this, the average value is the same in training and in evaluation, and nothing needs to be rescaled at test time.
- `model.train()` and `model.eval()`: Dropout is active only in train mode. In eval mode it passes values through unchanged. So `train_one_epoch` must call `model.train()` and `evaluate` must call `model.eval()`. Forgetting this is one of the most common bugs with Dropout.

In [ ]:
# TODO: write train_one_epoch(model, loader, loss_fn, optimizer)
#       start with model.train() so that Dropout is active
#       zero_grad -> forward -> loss -> backward -> step

In [ ]:
# TODO: write evaluate(model, loader, loss_fn) that returns (average loss, accuracy)
#       use model.eval() (this also switches Dropout off) and torch.no_grad()

In [ ]:
# TODO: write run_experiment(model) that:
#       - creates an Adam optimizer for this model
#       - for each epoch: train once, then evaluate on train and on val with evaluate()
#         (eval mode, so Dropout is off and the two losses can be compared)
#       - stores train loss and val loss for every epoch in two lists and returns them
#       - prints epoch number, train loss, val loss, val accuracy

## 5. Train both models

In [ ]:
# TODO: reset the seed, build Model A (no Dropout), move it to the device, and run the experiment

In [ ]:
# TODO: reset the seed, build Model B (with Dropout), move it to the device, and run the experiment

In [ ]:
# TODO: plot train loss and val loss vs epoch for both models
#       (two panels side by side, same y-axis range; add a title, axis labels and a legend)

In [ ]:
# TODO: Dropout experiment. Compute the validation accuracy of Model B three times:
#       once in eval mode, and twice in train mode (model.train(), still inside torch.no_grad())
#       hint: you cannot use evaluate() for the train-mode runs, because it switches to eval mode, so write a short loop
# TODO: print the three accuracies, then put Model B back in eval mode

#### Your answers: loss curves and Dropout

- Give the final train loss and validation loss of both models. Which model has the lower validation loss?
- Does Model A overfit? If yes, from which epoch? Compare the gap between train and validation loss at the last epoch for Model A and Model B. What did Dropout change?
- Write the three accuracies from the Dropout experiment. Why is the train-mode accuracy lower, and why do the two train-mode runs give different numbers?
- If you print the loss of each batch during training (Dropout on), Model B's training loss can be higher than its validation loss. Why is this not a bug?
- Was your prediction right?

Write your answers here.

## 6. Metrics on the validation set

Before you touch the test set, you compute all metrics on the validation set and use them to choose your best model. Write the metric functions once: in section 8 you use the same functions on the test set.

Key words, part 1 (one predicted class per image):
- Confusion matrix: a 10 × 10 table. Row = true class, column = predicted class. Cell `[i, j]` counts images of class `i` that the model predicted as class `j`.
- Precision of a class: of all images the model called "cat", how many really are cats. `precision = TP / (TP + FP)`
- Recall of a class: of all real cats, how many the model found. `recall = TP / (TP + FN)`
- F1 of a class: `2 · precision · recall / (precision + recall)`. It is high only when both are high.
- Macro F1: the plain average of the 10 per-class F1 scores. Every class counts the same.
- Weighted F1: the average of the per-class F1 scores, weighted by how many images each class has. Big classes count more.

Here TP (true positive) = real cat predicted as cat, FP (false positive) = not a cat but predicted as cat, FN (false negative) = real cat predicted as something else.

Key words, part 2 (scores and ranking):
- Score for class `c`: the softmax probability `p(c | image)`. A higher score means the model is more sure the image belongs to class `c`.
- Precision-recall (PR) curve: sort all images by their score for class `c`, highest first. Call the top `k` images "predicted c" and compute precision and recall. Every `k` gives one point (recall, precision).
- precision@k: the share of real class-`c` images among the top `k`.
- Average precision (AP) of class `c`: the average of precision@k over the ranks `k` where a real class-`c` image appears. `AP = (precision@k₁ + precision@k₂ + … + precision@k_P) / P`, where `P` is the number of real class-`c` images and `k₁, …, k_P` are their ranks. AP = 1.0 when all class-`c` images are ranked above all other images. AP uses no threshold: it measures how well the model ranks the images.
- Random baseline: a model that gives random scores gets AP of about the share of class `c` in the data. For a rare class this baseline is very small.
- Macro AP (also called mAP, mean average precision): the plain average of the 10 per-class APs.

Worked example for AP. Five images, sorted by their cat score:

| Rank k | 1 | 2 | 3 | 4 | 5 |
|---|---|---|---|---|---|
| Score for cat | 0.9 | 0.8 | 0.7 | 0.6 | 0.5 |
| Real cat? | yes | no | yes | no | no |
| precision@k | 1/1 | 1/2 | 2/3 | 2/4 | 2/5 |

The real cats are at ranks 1 and 3, so `AP = (1/1 + 2/3) / 2 = 0.833`. Your AP function must give 0.833 on this example.

In [ ]:
# TODO: write get_predictions(model, loader) that returns three tensors:
#       true labels, predicted labels (argmax), and probabilities of shape [N, 10] (softmax of the logits)
#       use model.eval() and torch.no_grad()

In [ ]:
# TODO: build the 10 x 10 confusion matrix yourself from the true and predicted labels
# TODO: from the confusion matrix, compute precision, recall and F1 for every class
#       if no image is predicted as a class, use precision = 0 for that class
# TODO: compute accuracy, macro F1 and weighted F1

In [ ]:
# TODO: write average_precision(is_positive, scores) yourself, following the definition above
#       is_positive: 1 for real class-c images, 0 for the rest; scores: the class-c probabilities
#       hint: sort by score (highest first), then use a cumulative sum of is_positive to get precision@k
# TODO: test it on the worked example: it must print 0.833
# TODO: compute AP for all 10 classes and the macro AP

In [ ]:
# TODO: apply your functions to the validation set for Model A and Model B
# TODO: print one table per model: one row per class with number of images, precision, recall, F1 and AP,
#       then accuracy, macro F1, weighted F1 and macro AP
# TODO: check your numbers with sklearn.metrics.classification_report and sklearn.metrics.average_precision_score,
#       they should match (tiny differences in AP can come from images with exactly the same score)

In [ ]:
# TODO: choose your best model: the one with the highest validation macro AP. Store it in best_model and print which one it is

#### Your answers: validation metrics

- Fill in for the validation set: | Model | accuracy | macro F1 | weighted F1 | macro AP |. Which model is your best model by macro AP? Do accuracy and macro F1 point to the same model?
- Why do we choose the best model on the validation set and not on the test set?

Write your answers here.

## 7. Threshold search on the validation set

So far your prediction was argmax: the class with the highest probability wins. Now look at one rare class at a time as a yes/no question, "is this a cat?", and use a threshold `t`: predict cat if `p(cat) ≥ t`. A lower `t` gives more cat predictions: recall usually goes up and precision usually goes down. The threshold search finds a good balance. Use `best_model` and the validation set only.

Key words:
- Decision threshold `t`: the score from which we answer "yes, this is a cat".
- Operating point: one chosen `t` together with its precision and recall. It is one point on the PR curve.
- Why not simply `t = 0.5`? With 10 classes, the winning class often has a probability well below 0.5. So "`p(cat) ≥ 0.5`" can be much stricter than argmax. Check this in your numbers.

Choose two thresholds for cat and two for dog, on the validation set:

| Rule | How to choose `t` |
|---|---|
| Rule F1 | The `t` with the highest validation F1 for that class. Call it `t_F1`. |
| Rule R70 | The app team wants to find at least 70% of all cats (or dogs). Among the `t` with validation recall ≥ 0.70, take the one with the highest validation precision. Call it `t_R70`. If no `t` reaches 0.70, take the `t` with the highest recall and say so. |

Search over the grid `t = 0.01, 0.02, …, 0.99`.

In [ ]:
# TODO: write prf_at_threshold(is_positive, scores, t) that returns (precision, recall, F1) for one class
#       an image is predicted positive if its score >= t; if no image is predicted positive, use precision = 0
# TODO: for cat and for dog: with best_model on the validation set, loop over the grid and store precision, recall and F1 for every t
# TODO: find t_F1 and t_R70 for cat and for dog, and print each one with its validation precision, recall and F1

In [ ]:
# TODO: plot precision, recall and F1 against t on the validation set (two panels: cat and dog)
#       add vertical lines at t = 0.5, t_F1 and t_R70, and a legend

#### Your answers: threshold search

- Write `t_F1` and `t_R70` for cat and dog. Are they above or below 0.5? Explain why, using what argmax does with 10 classes.
- From your plot: when `t` goes down, what happens to precision and to recall, and why? Why does F1 have a peak somewhere in the middle?
- By how much does validation recall for cat change when the model finds one more cat? What does this tell you about how exact your chosen thresholds are?

Write your answers here.

## 8. Final evaluation on the test set

All decisions are made. Now use the test set, once. Do not change your models, your choice of `best_model` or your thresholds after this point. If you do, say so in your answers.

In [ ]:
# TODO: get predictions on the test set for Model A and Model B
# TODO: with your own metric functions, print one table per model
#       (per class: number of images, precision, recall, F1, AP; then accuracy, macro F1, weighted F1, macro AP)
# TODO: check your numbers with sklearn.metrics.classification_report and average_precision_score

In [ ]:
# TODO: plot the test confusion matrix of both models with class names on both axes
#       hint: plt.imshow, or sklearn.metrics.ConfusionMatrixDisplay

In [ ]:
# TODO: for best_model, plot the test PR curves for cat and for dog (two panels), with the AP in the legend
#       hint: sklearn.metrics.precision_recall_curve, or build the curve yourself from the sorted scores
# TODO: draw the random baseline (share of cats, or dogs, in the test set) as a horizontal line
# TODO: mark four operating points on each curve: argmax (from the confusion matrix), t = 0.5, t_F1 and t_R70

In [ ]:
# TODO: for best_model, print one test table for cat and one for dog, one row per rule:
#       rule | t | images predicted positive | precision | recall | F1
#       rows: argmax, t = 0.5, t_F1, t_R70 (use the thresholds from section 7, do not tune them again)

#### Your answers: final metrics (explain properly)

1. Fill in for the test set: | Model | accuracy | macro F1 | weighted F1 | macro AP |. Which F1 is closer to accuracy, and why?
2. Explain each metric with your numbers. For the cat class of your best model, write one or two sentences for each of precision, recall, F1 and AP. Use counts from your test confusion matrix, for example: "Of the 80 images we predicted as cat, 30 were real cats, so precision = 30/80 = 0.38." (These numbers are made up, use yours.)
3. What does recall for cat and dog show that accuracy hides? How much accuracy would a model lose if it never predicted cat or dog at all?
4. Compute the random-baseline AP for cat and for airplane on the test set. Is your cat AP good or bad compared with its baseline? Why can the same AP value be good for one class and bad for another?
5. Changing `t` changed F1 for cat, but AP stayed the same. Why? When would you report AP, and when F1 at a chosen threshold?
6. Did Rule R70 still give recall ≥ 0.70 on the test set? Did Rule F1 give the best test F1 of the four rules? Explain any difference using what you know about the size of the validation set.
7. Did Dropout help on the test set? Compare Model A and Model B on macro F1, macro AP, and recall for cat and dog.
8. In the test confusion matrix of your best model, which classes are real cats most often predicted as? Does this make sense?

Write your answers here. A surprising result is fine if you explain it with your numbers.

## 9. Group contribution statement (required)

For each member, write what they did (e.g. data subsets, CNN design, training loop, Dropout experiment, metric functions, threshold search, plots, answers) and how the group used AI (what for, and how you checked it). Every member must still be able to explain the whole notebook.

| Student (full name) | Main contribution | How AI was used and checked |
|---|---|---|
|  |  |  |
|  |  |  |
|  |  |  |

## 10. Defense checklist and grading

Submit this completed notebook (or an HTML/PDF export with all cells and plots visible) before your defense. Each group has a short (~5 min) defense. Any member may be asked to:

- explain any line of your code: the subsets, every layer of your CNN, and the training steps
- show how the image shape changes through your CNN, and how you got the input size of the first `Linear` layer
- compute the number of parameters of one of your `Conv2d` layers by hand
- explain what Dropout does in train mode and in eval mode, and show the lines in your code where the mode is switched
- explain precision and recall for one class using a row and a column of your confusion matrix
- compute AP by hand for a short ranked list given by the instructor
- explain why the thresholds were chosen on the validation set and not on the test set, and what happens to precision and recall when `t` goes down
- explain why accuracy can look fine while the model fails on cats and dogs
- say what AI helped with and how you checked it

The Week 7 SIS mini-project is worth 4 points:

| Criterion | Full credit requires | Points |
|---|---|---|
| 1. Your CNN and training | Code written by the group runs from top to bottom. The three subsets match the table. The CNN follows every rule in section 2 and is built from PyTorch layers, not a ready-made or pretrained network. Model A and Model B differ only by Dropout. `model.train()` and `model.eval()` are used correctly. Both models are trained with the same settings, and the loss curves of both are plotted with the same y-axis range, a title, axis labels and a legend. | 1 |
| 2. Metrics and threshold search | Confusion matrix, per-class precision, recall, F1 and AP are computed by your own functions and match scikit-learn. The AP function gives 0.833 on the worked example. The best model and all four thresholds are chosen on the validation set only. Plots: confusion matrices with class names, precision/recall/F1 against `t`, and test PR curves with the random baseline and the four operating points. | 1 |
| 3. Explanations | All answer cells are filled in, correct, short, and use your own numbers: Dropout and train vs eval mode, each metric explained for the cat class, AP vs F1 and the random baseline, the threshold trade-off, and how the thresholds worked on the test set. | 1 |
| 4. Oral defense | Each member individually explains the group's code, plots and metrics (checklist above). | 1 |
| Total | | 4 |

How points are given:
- Criteria 1 to 3 are the same for the whole group. Criterion 4 is given to each student separately.
- A criterion that is mostly met, with one listed part missing or wrong, gets 0.5.
- If the test set was used to choose the model or a threshold, criterion 2 gets at most 0.5.
- If the CNN is a ready-made or pretrained network, criterion 1 gets 0.

Optional (no extra points):
- Class weights: pass `weight=` to `nn.CrossEntropyLoss` so rare classes count more (for example 10 for cat and dog, 1 for the rest). Train Model B again. Did recall for cat and dog go up? What happened to accuracy and to the best thresholds?
- ROC-AUC vs AP: for "cat vs not cat" on the test set, compute ROC-AUC with `sklearn.metrics.roc_auc_score` and plot the ROC curve with `sklearn.metrics.RocCurveDisplay`. A random model gets ROC-AUC 0.5 but AP equal to the share of cats. Why does ROC-AUC look much better than AP for a rare class?
- Rare vs hard: cats and dogs are hard classes even in normal, balanced CIFAR-10. How could you check how much of their low recall comes from having few training images, and how much from the classes being hard?
- Thresholds inside the 10-class classifier: predict cat if `p(cat) ≥ t_F1`, otherwise use argmax. What happens to accuracy and to the recall of the other classes?